# Silver: typed variant reference

Flattens the verbatim ClinVar records into one queryable row per variant. Bronze holds
nested JSON that neither SQL nor an agent can interrogate; this is where it becomes
answerable.

| | |
| --- | --- |
| **Reads** | `bronze_clinvar_variants`, `bronze_reference_coverage` |
| **Writes** | `silver_variant_reference`, `silver_field_quality` |

### The populated-rate gate

Every field the consultation report is permitted to cite is asserted here. A field that
is unexpectedly empty across the whole run **fails the notebook** rather than
publishing. This exists because a silently-null attribute is indistinguishable from a
real negative once it reaches prose, and it will be read as one.

In [ ]:
PIPELINE_RUN_ID = ""

In [ ]:
import json
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F

# --- cross-lakehouse reads -------------------------------------------------------
# spark.read.table() only resolves against this notebook's default lakehouse, so any
# table in a different layer is read by explicit OneLake path.
_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table_name):
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    path = (f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table_name}")
    return spark.read.format("delta").load(path)


from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

bronze = lake_table("bronze_lakehouse", "bronze_clinvar_variants")
RUN_ID = PIPELINE_RUN_ID or (bronze.orderBy(F.desc("ingested_at_utc"))
                             .select("run_id").first()["run_id"])
GENERATED_AT = datetime.now(timezone.utc).isoformat()
print("run_id:", RUN_ID)

## 1. Parse the nested record

ClinVar's summary shape varies by record age and submission route, so every field is
read defensively. A missing field becomes null and is counted, never guessed.

In [ ]:
# ACMG/AMP review status maps to a confidence ordinal. A single submitter with no stated
# assertion criteria is not equivalent to an expert panel, and flattening that difference
# would overstate certainty in the report.
REVIEW_RANK = {
    "practice guideline": 4,
    "reviewed by expert panel": 3,
    "criteria provided, multiple submitters, no conflicts": 2,
    "criteria provided, conflicting classifications": 1,
    "criteria provided, conflicting interpretations": 1,
    "criteria provided, single submitter": 1,
    "no assertion criteria provided": 0,
    "no classification provided": 0,
    "no assertion provided": 0,
}

# The five-tier ACMG/AMP scale. "Uncertain" is its own state, never a midpoint that can
# be rounded toward benign.
TIER = {
    "pathogenic": "Pathogenic",
    "likely pathogenic": "Likely pathogenic",
    "pathogenic/likely pathogenic": "Pathogenic",
    "uncertain significance": "Uncertain significance",
    "conflicting classifications of pathogenicity": "Conflicting",
    "conflicting interpretations of pathogenicity": "Conflicting",
    "likely benign": "Likely benign",
    "benign": "Benign",
    "benign/likely benign": "Benign",
}


def classify(description):
    if not description:
        return None, None
    key = description.strip().lower()
    tier = TIER.get(key)
    if tier is None:
        for candidate, mapped in TIER.items():
            if candidate in key:
                tier = mapped
                break
    return tier, description.strip()


def first_variation(record):
    variations = record.get("variation_set") or []
    return variations[0] if variations else {}


def parse(record_json, gene_symbol, accession, uid):
    record = json.loads(record_json)
    germline = record.get("germline_classification") or {}
    variation = first_variation(record)

    tier, raw_significance = classify(germline.get("description"))
    review_status = (germline.get("review_status") or "").strip() or None

    conditions, condition_ids = [], []
    for trait in (germline.get("trait_set") or []):
        name = trait.get("trait_name") or trait.get("name")
        if name:
            conditions.append(name)
        for xref in (trait.get("trait_xrefs") or []):
            source, identifier = xref.get("db_source"), xref.get("db_id")
            if source and identifier:
                condition_ids.append(f"{source}:{identifier}")

    consequences = [c.get("type") for c in (record.get("molecular_consequence_list") or [])
                    if isinstance(c, dict) and c.get("type")]
    if not consequences:
        consequences = [c for c in (record.get("molecular_consequence_list") or [])
                        if isinstance(c, str)]

    return {
        "run_id": RUN_ID,
        "variant_key": f"{gene_symbol}:{accession}",
        "accession": accession,
        "clinvar_uid": uid,
        "gene_symbol": gene_symbol,
        "variant_title": record.get("title"),
        "hgvs_c": (variation.get("cdna_change")
                   or (record.get("title") or "").split(" ")[0] or None),
        "protein_change": record.get("protein_change") or variation.get("protein_change"),
        "variant_type": variation.get("variant_type") or record.get("obj_type"),
        "chromosome": variation.get("chr"),
        "clinical_significance": tier,
        "clinical_significance_raw": raw_significance,
        "review_status": review_status,
        "review_confidence": REVIEW_RANK.get((review_status or "").lower()),
        "last_evaluated": (germline.get("last_evaluated") or "").strip() or None,
        "condition_names": "; ".join(dict.fromkeys(conditions))[:600] or None,
        "condition_identifiers": "; ".join(dict.fromkeys(condition_ids))[:600] or None,
        "molecular_consequence": "; ".join(dict.fromkeys(consequences))[:300] or None,
        "record_status": record.get("record_status"),
        "generated_at_utc": GENERATED_AT,
    }


rows = []
for row in (bronze.filter(F.col("run_id") == RUN_ID)
            .select("record_json", "gene_symbol", "accession", "uid").collect()):
    try:
        rows.append(parse(row["record_json"], row["gene_symbol"],
                          row["accession"], row["uid"]))
    except Exception as error:
        print(f"  parse failed for {row['accession']}: {str(error)[:110]}")

print(f"parsed {len(rows)} variants")

In [ ]:
VARIANT_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("variant_key", StringType(), True),
    StructField("accession", StringType(), True),
    StructField("clinvar_uid", StringType(), True),
    StructField("gene_symbol", StringType(), True),
    StructField("variant_title", StringType(), True),
    StructField("hgvs_c", StringType(), True),
    StructField("protein_change", StringType(), True),
    StructField("variant_type", StringType(), True),
    StructField("chromosome", StringType(), True),
    StructField("clinical_significance", StringType(), True),
    StructField("clinical_significance_raw", StringType(), True),
    StructField("review_status", StringType(), True),
    StructField("review_confidence", IntegerType(), True),
    StructField("last_evaluated", StringType(), True),
    StructField("condition_names", StringType(), True),
    StructField("condition_identifiers", StringType(), True),
    StructField("molecular_consequence", StringType(), True),
    StructField("record_status", StringType(), True),
    StructField("generated_at_utc", StringType(), True),
])


def write_run_scoped(dataframe, table_name):
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (dataframe.write.format("delta").mode("overwrite")
            .partitionBy("run_id").saveAsTable(table_name))
    except Exception as error:
        print(f"  WARNING: {table_name} schema changed - replacing all runs "
              f"({str(error).splitlines()[0][:110]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


columns = [field.name for field in VARIANT_SCHEMA.fields]
variants = spark.createDataFrame(
    [tuple(row.get(column) for column in columns) for row in rows],
    schema=VARIANT_SCHEMA)
write_run_scoped(variants, "silver_variant_reference")
print(f"silver_variant_reference: {variants.count():,} rows")

## 2. Populated-rate gate

`REQUIRED` fields are those the report may cite. If any is empty across the entire run,
the notebook raises and nothing downstream is published — an empty field must never
reach prose as an absence of finding.

In [ ]:
REQUIRED = {
    "accession": 0.99,
    "gene_symbol": 0.99,
    "variant_title": 0.95,
    "clinical_significance": 0.80,
    "review_status": 0.80,
}
OBSERVED = ["protein_change", "condition_names", "condition_identifiers",
            "molecular_consequence", "hgvs_c", "last_evaluated"]

total = variants.count()
quality_rows, failures = [], []

for column in [field.name for field in VARIANT_SCHEMA.fields]:
    filled = variants.filter(F.col(column).isNotNull() & (F.col(column) != "")).count()
    rate = (filled / total) if total else 0.0
    minimum = REQUIRED.get(column)
    status = "ok"
    if minimum is not None and rate < minimum:
        status = "FAILED"
        failures.append(f"{column}: {rate:.1%} populated, requires {minimum:.0%}")
    elif column in OBSERVED and filled == 0:
        # Not fatal, but it must be visible: a field that is always empty is a field the
        # report will silently omit, and someone will assume it was checked.
        status = "empty"
    quality_rows.append({
        "run_id": RUN_ID, "table_name": "silver_variant_reference",
        "column_name": column, "populated": filled, "total": total,
        "populated_rate": round(rate, 4),
        "required_rate": minimum, "status": status,
        "checked_at_utc": GENERATED_AT,
    })

QUALITY_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("populated", IntegerType(), True),
    StructField("total", IntegerType(), True),
    StructField("populated_rate", DoubleType(), True),
    StructField("required_rate", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("checked_at_utc", StringType(), True),
])
quality_columns = [field.name for field in QUALITY_SCHEMA.fields]
write_run_scoped(
    spark.createDataFrame(
        [tuple(row.get(column) for column in quality_columns) for row in quality_rows],
        schema=QUALITY_SCHEMA),
    "silver_field_quality")

for row in sorted(quality_rows, key=lambda r: r["populated_rate"]):
    flag = {"ok": "  ", "empty": "!!", "FAILED": "XX"}[row["status"]]
    print(f" {flag} {row['column_name']:26} {row['populated_rate']:7.1%}  "
          f"{row['populated']:>5}/{row['total']}")

if failures:
    raise ValueError("Field quality gate failed:\n  " + "\n  ".join(failures))
print("\nfield quality gate: PASSED")

In [ ]:
display(spark.read.table("silver_variant_reference")
        .filter(F.col("run_id") == RUN_ID)
        .groupBy("gene_symbol", "clinical_significance").count()
        .orderBy("gene_symbol", "clinical_significance"))